# Figure S6 -- wavenumber-frequency spectra

Variance-preserving spectra of hourly sea surface height over one forecast cycle,
before and after the tide is removed, for each of the three regions.

The archived spectra are stored on the axes the figure plots -- frequency in cycles
per hour and wavenumber in cycles per kilometre -- so the shaded quantity is
recovered directly as

```python
frequency * wavenumber * E * 1e4     # cm^2
```

Overlaid are the M2 and K1 tidal frequencies, the local inertial frequency, the
cutoff of the detiding filter, and the first two internal-wave dispersion curves
from `omega^2 = f^2 + (c*kappa)^2`. Every one of those values is an attribute of
the file, so the overlays need no outside information.

**What this reproduces.** The whole figure; these spectra are computed entirely
from the forecast.


In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Folder holding the archived NetCDF files. Default: alongside this notebook.
DATA = Path(".")

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

REGION_ORDER = ["Kuroshio", "Gulf of Mexico", "Arabian Sea"]


def by_region(paths):
    """Open each file, read its region attribute, return them in paper order."""
    tagged = []
    for p in paths:
        with xr.open_dataset(p) as ds:
            tagged.append((ds.attrs.get("region", p.stem), p))
    key = lambda t: (REGION_ORDER.index(t[0]) if t[0] in REGION_ORDER else 99, t[0])
    return sorted(tagged, key=key)


def need(pattern):
    found = sorted(DATA.glob(pattern))
    if not found:
        raise FileNotFoundError(
            f"No file matching {pattern!r} in {DATA.resolve()}. "
            f"Set DATA to the folder holding the archived NetCDF files.")
    return found


In [ ]:
import matplotlib.colors as colors

XLIM = (1 / 1500.0, 1 / 8.0)      # cpkm
DYN  = 1e4                        # colour range: VMAX/DYN .. VMAX

files = by_region(need("FigureS6_spectra_*.nc"))

# one colour scale across every panel, set by the tided spectra
VMAX = 0.0
for region, path in files:
    with xr.open_dataset(path) as ds:
        vp = (ds.frequency.values[:, None] * ds.wavenumber.values[None, :]
              * ds.E_raw.values) * 1e4
        VMAX = max(VMAX, float(np.nanpercentile(vp, 99.5)))
norm = colors.LogNorm(vmin=VMAX / DYN, vmax=VMAX)

fig, axes = plt.subplots(2, len(files), figsize=(4.3 * len(files), 8.2),
                         dpi=130, sharex=True, sharey=True, squeeze=False)

for c, (region, path) in enumerate(files):
    with xr.open_dataset(path) as ds:
        kr, w = ds.wavenumber.values, ds.frequency.values
        fcp   = ds.attrs["inertial_frequency_cph"]
        ylim  = (float(w.min()), 0.5)
        for r, (var, rowname) in enumerate([("E_raw", "Raw SSH"),
                                            ("E_detided", "Detided SSH")]):
            ax = axes[r][c]
            vp = (w[:, None] * kr[None, :] * ds[var].values) * 1e4
            pcm = ax.pcolormesh(kr, w, vp, norm=norm, cmap="turbo",
                                shading="auto")
            ax.set_xscale("log"); ax.set_yscale("log")
            ax.set_xlim(XLIM); ax.set_ylim(ylim)

            for y, txt, ls in [(ds.attrs["M2_frequency_cph"], "M2", "-"),
                               (ds.attrs["K1_frequency_cph"], "K1", "-"),
                               (fcp, "f", ":"),
                               (1.0 / ds.attrs["detiding_filter_period_hours"],
                                f"{ds.attrs['detiding_filter_period_hours']:.0f} h",
                                "--")]:
                ax.axhline(y, color="w", lw=0.8, ls=ls)
                ax.text(XLIM[1] * 0.93, y * 1.06, txt, color="w", fontsize=7,
                        ha="right", va="bottom")

            kk = np.logspace(np.log10(XLIM[0]), np.log10(XLIM[1]), 200)  # cpkm
            for cn, lsty in ((ds.attrs["internal_wave_phase_speed_mode1_m_s"], "-"),
                             (ds.attrs["internal_wave_phase_speed_mode2_m_s"], "--")):
                wcp = np.sqrt((fcp / 3600.0) ** 2 + (cn * kk / 1e3) ** 2) * 3600.0
                ax.plot(kk, wcp, color="k", lw=1.0, ls=lsty)

            ax.tick_params(labelsize=8)
            ax.text(0.03, 0.96, f"({chr(97 + r * len(files) + c)})",
                    transform=ax.transAxes, fontsize=8, fontweight="bold",
                    color="w", va="top")
            if r == 0:
                ax.set_title(region, fontsize=11, fontweight="bold", loc="left")
            if c == 0:
                ax.set_ylabel(f"{rowname}\nFrequency [cph]", fontsize=9)
            if r == 1:
                ax.set_xlabel("Horizontal wavenumber [cpkm]", fontsize=9)

fig.subplots_adjust(left=0.08, right=0.88, top=0.94, bottom=0.08,
                    wspace=0.06, hspace=0.08)
cax = fig.add_axes([0.90, 0.20, 0.015, 0.6])
cb = fig.colorbar(pcm, cax=cax, extend="both")
cb.set_label(r"$\omega\,\kappa\,E(\kappa,\omega)$  [cm$^2$]", fontsize=9)
cb.ax.tick_params(labelsize=8)

out = FIGDIR / "FigureS6_omega_k_spectra.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("wrote", out)


In [ ]:
# ---- frequency spectra, integrated over wavenumber -----------------------
fig, axes = plt.subplots(1, len(files), figsize=(4.3 * len(files), 3.6),
                         dpi=130, sharey=True, squeeze=False)

for ax, (region, path) in zip(axes[0], files):
    with xr.open_dataset(path) as ds:
        kr, w = ds.wavenumber.values, ds.frequency.values
        dk = np.gradient(kr)
        for var, col, lab in (("E_raw", "k", "raw"),
                              ("E_detided", "r", "detided")):
            Ew = np.nansum(ds[var].values * dk[None, :], axis=1)   # m2 / cph
            ax.loglog(w, Ew * 1e4, color=col, lw=1.4, label=lab)   # cm2 / cph
        for y, txt in ((ds.attrs["M2_frequency_cph"], "M2"),
                       (ds.attrs["K1_frequency_cph"], "K1"),
                       (ds.attrs["inertial_frequency_cph"], "f")):
            ax.axvline(y, color="0.4", lw=0.8, ls=":")
            ax.text(y, ax.get_ylim()[1], txt, fontsize=7, ha="center",
                    va="bottom")
        ax.set_xlim(float(w.min()), 0.5)
        ax.set_title(region, fontsize=10, fontweight="bold", loc="left")
        ax.set_xlabel("Frequency [cph]", fontsize=9)
        ax.grid(True, which="both", ls="--", alpha=0.3)
        ax.tick_params(labelsize=8)

axes[0][0].set_ylabel(r"$E(\omega)$  [cm$^2$ cph$^{-1}$]", fontsize=9)
axes[0][0].legend(fontsize=8)
fig.tight_layout()
out = FIGDIR / "FigureS6_frequency_spectra.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("wrote", out)


In [ ]:
# ---- the hourly fields the spectra came from, if they are in the deposit --
inputs = sorted(DATA.glob("FigureS6_input_ssh_*.nc"))
if not inputs:
    print("hourly input fields not in this deposit -- skipping")
else:
    fig, axes = plt.subplots(len(inputs), 1, figsize=(10, 2.4 * len(inputs)),
                             dpi=120, sharex=True, squeeze=False)
    for ax, (region, path) in zip(axes[:, 0], by_region(inputs)):
        with xr.open_dataset(path) as ds:
            j, i = ds.latitude.size // 2, ds.longitude.size // 2
            t_d = (ds.time.values - ds.time.values[0]) / np.timedelta64(1, "D")
            ax.plot(t_d, ds.ssh_tided[:, j, i] * 100, "k", lw=0.8, label="raw")
            ax.plot(t_d, ds.ssh[:, j, i] * 100, "r", lw=1.5, label="detided")
            ax.set_ylabel("SSH [cm]", fontsize=9)
            ax.set_title(region, fontsize=10, loc="left")
            ax.grid(ls="--", alpha=0.3)
    axes[0][0].legend(fontsize=8)
    axes[-1][0].set_xlabel("Forecast time [days]", fontsize=9)
    fig.tight_layout()
    out = FIGDIR / "FigureS6_box_centre_timeseries.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("wrote", out)
